# Deep heads on a regularised tree encoding — MagicTelescope

**The question.** The feature side is settled: deleting the deepest tree layers stops the model memorising, on every dataset where it was memorising. Your advisor's next step is to keep that setup and swap the linear head for a deep network, then ask whether the network's extra capacity turns into **better predictions** or into **a new way to overfit** — and whether the network needs regularising of its own on top of the regularised features.

### Why MagicTelescope
It is the binary dataset where the linear head fell clearly short of the trees, so a deep head has room to show a gain larger than the measurement noise.

| dataset | best tree | linear head on bits | shortfall |
|---|---|---|---|
| credit | 0.8453 | 0.8457 | none left, already matched |
| house_16H | 0.9526 | 0.9441 | 0.0085, inside the noise |
| **MagicTelescope** | **0.9377** | **0.9047** | **0.0329, three times the noise** |

The targets form a ladder: beat the linear head at **0.9047**, reach the tree at **0.9377**. For context, an earlier run put a deep network on this dataset's *raw* features and reached 0.9441, but adding the full tree encoding to that network made it *worse* (0.9241). That is the memorisation problem in miniature, on this very dataset, and it is what Part C re-tests now that the features are clean.

### The heads
| head | what it is |
|---|---|
| **linear** | logistic regression, penalty chosen on validation. The reference every deep head must beat. |
| **mlp** | a plain multi-layer network on the bits |
| **tabresnet** | the residual network used earlier in this project |
| **treetf** | a transformer whose tokens are **trees**, not bits |

**About the transformer.** A transformer lets every input attend to every other, at a cost growing with the square of the token count, so 1,450 bits would be far too slow. But each bit belongs to exactly one tree, so we bundle each tree's bits into a single token. The transformer then runs over 100 tokens and learns how trees relate to one another. The bits-to-token projection is shared across trees, because trees are interchangeable, and a learned per-tree embedding lets the model still tell them apart.

### The grid
| part | what runs | why |
|---|---|---|
| **A** | the linear reference, on both encodings | so a deep head cannot win merely by being better tuned |
| **B** | 3 heads × 3 sizes × 4 regularisation settings on the **shallow** bits (36 runs) | the main question |
| **C** | the same heads and sizes on the **full** encoding, 2 settings (18 runs) | capacity versus generalisation |

Regularisation is a clean 2×2 so every cell is interpretable: none, dropout only, weight decay only, both. Everything else is held fixed: same forest, same honest bits, same split, same optimiser and learning rate, up to 150 epochs, stopping when validation has not improved for 30, and the best-validation epoch is the one reported.

### If your runtime disconnects
Nothing is lost and the second run is fast. The forest and the bits are cached on Drive, so they are computed once ever. Each finished run is copied to Drive immediately and skipped on a rerun. And inside a run, training checkpoints to Drive every 10 epochs, so an interrupted run continues from where it stopped rather than restarting. Just re-run the notebook from the top.

⏱ **about 2 to 3 hours on a T4** for all three parts. Parts B and C are separate cells, so you can stop after B and look at the results.

In [ ]:
# 1 · check the GPU
import torch
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NO GPU — set Runtime > Change runtime type > T4 GPU')

In [ ]:
# 2 · mount Drive and lay out the folders that make a disconnect harmless
from google.colab import drive
drive.mount('/content/drive')
import os, glob, json, shutil
DRIVE = '/content/drive/MyDrive/TKCE/deep_head'
CACHE = f'{DRIVE}/cache'    # the forest and the bits: computed once, ever
CKPT  = f'{DRIVE}/ckpt'     # mid-training checkpoints, every 10 epochs
RUNS  = f'{DRIVE}/runs'     # each finished run, copied over the moment it ends
for d in (CACHE, CKPT, RUNS): os.makedirs(d, exist_ok=True)
print('Drive is ready at', DRIVE)

In [ ]:
# 3 · get the code
%cd /content
!git clone https://github.com/sushanedulloo/TKCE.git 2>/dev/null || echo 'already cloned'
%cd /content/TKCE
!git pull
!pip install -q openml catboost optuna lightgbm xgboost

In [ ]:
# 4 · restore anything an earlier session already finished
ROOT = 'results/deep_head'
os.makedirs(ROOT, exist_ok=True)
if glob.glob(f'{RUNS}/*/*/deephead_*.json'):
    shutil.copytree(RUNS, ROOT, dirs_exist_ok=True)
    done = glob.glob(f'{ROOT}/*/*/deephead_*.json')
    print(f'restored {len(done)} finished runs from Drive — they will be skipped')
else:
    print('no earlier runs on Drive — starting fresh')
ck = glob.glob(f'{CKPT}/ck_*.pt')
print(f'{len(ck)} mid-training checkpoint(s) waiting to be continued'
      if ck else 'no mid-training checkpoints')
SUITE = (f'--root {ROOT} --cache-dir {CACHE} --ckpt-dir {CKPT} --sync-dir {RUNS}')
print('\n' + SUITE)

## Part A — the linear reference
Logistic regression on both encodings, with the penalty swept and chosen on validation. This is the bar every deep head has to clear. Two runs, about a minute.

In [ ]:
!python -u run_deep_head_suite.py {SUITE} --part A 2>&1 | tee -a {ROOT}/suite.log

## Part B — deep heads on the shallow encoding
36 runs: three heads, three sizes, four regularisation settings. This is the main experiment. About 1.5 to 2 hours on a T4. Safe to interrupt and rerun.

In [ ]:
!python -u run_deep_head_suite.py {SUITE} --part B 2>&1 | tee -a {ROOT}/suite.log

## Part C — the same heads on the full encoding
18 runs. This is the capacity-versus-generalisation comparison: if the big heads help on the shallow bits but hurt on the full ones, then feature regularisation is what lets a network use its capacity. About 45 minutes.

In [ ]:
!python -u run_deep_head_suite.py {SUITE} --part C 2>&1 | tee -a {ROOT}/suite.log

In [ ]:
# 5 · the summary table: best run per head, chosen on validation, never on test
!python -u run_deep_head_suite.py {SUITE} --aggregate-only
import pandas as pd
pd.set_option('display.width', 250)
b = pd.read_csv(f'{ROOT}/suite_summary.csv')
print(b.to_string(index=False))

In [ ]:
# 6 · did any deep head beat the linear one, and did it stay honest?
t = pd.read_csv(f'{ROOT}/suite_long.csv')
lin = t[(t['head'] == 'linear') & (t.bits == 'shallow')].test.iloc[0]
ceil = t.tree_ceiling.iloc[0]
sh = t[(t.bits == 'shallow') & (t['head'] != 'linear')]
print(f'linear reference {lin:.4f}   |   best tree {ceil:.4f}   |   '
      f'scores are uncertain by about 0.01 here\n')
beat = sh[sh.test > lin + 0.01]
print(f'deep runs clearly beating the linear head: {len(beat)} of {len(sh)}')
if len(beat):
    print(beat[['head','size','reg','n_params','train','val','test','gap']]
          .sort_values('test', ascending=False).round(4).to_string(index=False))
print('\n--- best per head, and what regularisation did to it ---')
for h in ['mlp','tabresnet','treetf']:
    g = sh[sh['head'] == h]
    if not len(g): continue
    nr, br = g[g.reg == 'none'], g.sort_values('val').iloc[-1]
    print(f'  {h:10s} best {br.test:.4f} ({br["size"]}, {br.reg}, gap {br.gap:+.4f})'
          f'   vs linear {br.test - lin:+.4f}   vs tree {br.test - ceil:+.4f}')
    if len(nr):
        w = nr.sort_values('val').iloc[-1]
        print(f'             unregularised best {w.test:.4f} (gap {w.gap:+.4f})')

In [ ]:
# 7 · the charts
from IPython.display import Image, display
for f in ['fig1_heads.png', 'fig2_capacity.png', 'fig3_curves.png']:
    p = f'{ROOT}/{f}'
    if os.path.exists(p):
        print('=' * 30, f, '=' * 30); display(Image(p))

In [ ]:
# 8 · BUNDLE AND DOWNLOAD everything: logs, results, tables, charts, README
#     (re-aggregates first, so the bundle is complete even after a partial session)
!python -u run_deep_head_suite.py {SUITE} --aggregate-only --bundle 2>&1 | tail -8
zips = sorted(glob.glob('results/deep_head_*.zip'), key=os.path.getmtime)
z = zips[-1]
shutil.copy(z, f'{DRIVE}/{os.path.basename(z)}')
shutil.copytree(ROOT, RUNS, dirs_exist_ok=True)
print(f'\n{os.path.basename(z)}   {os.path.getsize(z)/1e6:.1f} MB')
print('also copied to Drive:', f'{DRIVE}/{os.path.basename(z)}')
print('\ncontents:')
!unzip -l {z} | tail -n +4 | head -70
from google.colab import files
files.download(z)